# ZapWise: train the container model on your own photos

This notebook retrains the vision model so it recognises **container types** (not just material families).
It uses the same script as the baseline (`train_vision.py`), so the exported file drops straight into the app.

**Folder layout** (zip it as `containers.zip`, one folder per container key from `ZapWise_DB.xlsx` > Containers):

```
containers/
  cpet_tray/      obj01_1.jpg obj01_2.jpg ... obj02_1.jpg ...
  pp_container/   ...
  paperboard_tray/
  paper_cup/
  steam_bag/
  glass/
  ceramic/
  metal/
  eps_foam/
  pet_clamshell/
```

**Photo protocol**
- At least 15 different physical objects per class, 6 to 10 photos per object.
- Name files `<objectid>_<n>.jpg`. Photos of one object stay together in train, validation or test, so the score is not inflated by near-duplicates.
- Vary angle, distance, light (window, ceiling lamp), background (counter, table, hand) and whether food is inside.
- Keep a written log: date, store, product, container marking (resin code, 'oven safe').

## 1. Get the code and the photos

In [ ]:
# Option A: your GitHub repo (replace the URL)
# !git clone https://github.com/<you>/zapwise.git
# %cd zapwise/ml

# Option B: upload ml.zip (the ml folder) and containers.zip with the file panel on the left, then:
!unzip -q -o ml.zip -d /content/ && unzip -q -o containers.zip -d /content/photos/
%cd /content/ml
!ls /content/photos/containers

## 2. Train
`--weights imagenet` lets Keras download the MobileNetV2 ImageNet weights (Colab has internet). `--app-map identity` keeps class names as container keys.

In [ ]:
!python train_vision.py --data /content/photos/containers --out /content/out/containers --weights imagenet --aug 3 --app-map identity

## 3. Check the result
Look at per-class F1 and the confusion matrix. Classes that are often confused need more objects, not more photos of the same object.

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open('/content/out/containers/metrics.json'))
print('test accuracy', round(m['test_acc'], 3), 'macro F1', round(m['test_macro_f1'], 3))
print(json.dumps(m['per_class_f1'], indent=1))
display(Image('/content/out/containers/confusion.png'))

## 4. Product look-up check (optional)
Put 3 to 6 photos per product in `/content/photos/products/<product_id>/` and measure how well re-photographed packages are found.

In [ ]:
# !python retrieval_eval.py --data /content/photos/products --model /content/out/containers/zapwise_vision.tflite --out /content/out/retrieval
# !python build_catalog.py --photos /content/photos/products --model /content/out/containers/zapwise_vision.tflite --products products.json --out /content/out/catalog_embeddings.json

## 5. Download and put into the app
- `zapwise_vision.tflite` -> `app/assets/models/zapwise_vision.tflite`
- `labels.json` -> `app/assets/models/vision_labels.json`
- `catalog_embeddings.json` -> `app/assets/data/catalog_embeddings.json`
- After replacing the model, run `ml/build_catalog.py` again, because catalog vectors must come from the same model.

In [ ]:
from google.colab import files
files.download('/content/out/containers/zapwise_vision.tflite')
files.download('/content/out/containers/labels.json')